<a href="https://colab.research.google.com/github/iMaIrsdyh/BigData26_B_2411533018_KarimahIrsyadiyah/blob/main/Praktikum4/BD_P04_2411533018_Karimah_Irsyadiyah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**KARIMAH IRSYADIYAH**

2411533015

**PRAKTIKUM 4**

## K-1. Menyiapkan Lingkungan dan Data Contoh

Tahap ini dilakukan untuk menyiapkan lingkungan praktikum, memasang pustaka yang dibutuhkan, dan membaca dataset `trips_bersih/` dari hasil Praktikum 3.

Dataset kemudian disiapkan dengan mengganti beberapa nama kolom asli menjadi nama yang lebih mudah dipahami. Selanjutnya, diambil sampel maksimal 300.000 baris untuk percobaan pada K-2 dan K-3 agar proses pengujian lebih ringan.

Pustaka yang digunakan meliputi pandas dan NumPy untuk pengolahan data, SQLite untuk penyimpanan relasional, serta pustaka DuckDB, NetworkX, Shapely, TinyDB, dan Delta Lake untuk tahapan praktikum berikutnya.

In [14]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

from google.colab import drive
drive.mount("/content/drive")

import os
import zipfile

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"

# Lokasi file ZIP dari Praktikum 3
ZIP_PATH = f"{DIR_P3}/trips_bersih.zip"

# Folder dataset untuk Praktikum 4
TRIPS_BERSIH = f"{DIR_SIMPAN}/trips_bersih"

os.makedirs(DIR_SIMPAN, exist_ok=True)

# Ekstrak ZIP jika folder belum tersedia
if not os.path.exists(TRIPS_BERSIH):
    os.makedirs(TRIPS_BERSIH, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(TRIPS_BERSIH)

assert os.path.exists(TRIPS_BERSIH), (
    "Folder trips_bersih/ tidak ditemukan."
)

print("Dataset tersedia:", TRIPS_BERSIH)
print("Jumlah folder partisi:", len(os.listdir(TRIPS_BERSIH)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset tersedia: /content/drive/MyDrive/BigData/Praktikum4/trips_bersih
Jumlah folder partisi: 8


### Penjelasan Kode

1. `ukur()` digunakan untuk mengukur waktu pembacaan dataset menggunakan `time.perf_counter()`.
2. `pd.read_parquet()` digantikan dengan `pyarrow.dataset` agar struktur partisi Hive dapat dibaca, termasuk partisi yang memiliki nilai kosong.
3. `NAMA_BARU` digunakan untuk mengganti nama kolom asli TLC menjadi nama yang lebih mudah dipahami.
4. `astype(str)` mengubah kolom wilayah menjadi tipe string.
5. `sample()` mengambil maksimal 300.000 baris secara acak untuk percobaan K-2 dan K-3.
6. `random_state=42` digunakan agar sampel tetap konsisten ketika kode dijalankan kembali.
7. `id_perjalanan` memberikan identitas unik untuk setiap baris sampel.

### Hasil K-1

Persiapan lingkungan praktikum berhasil dilakukan. Pustaka yang dibutuhkan telah dipasang dan Google Drive berhasil dihubungkan.

Dataset `trips_bersih/` dari Praktikum 3 berhasil dibaca menggunakan PyArrow. Beberapa nama kolom asli telah diubah menjadi nama yang lebih mudah dipahami.

Waktu pembacaan dataset dicatat menggunakan fungsi `ukur()`. Selanjutnya, diambil sampel maksimal 300.000 baris dengan `random_state=42` untuk digunakan pada percobaan K-2 dan K-3.

In [15]:

import pyarrow as pa
import pyarrow.dataset as ds
import pandas as pd
import time
import os
import sqlite3
import numpy as np

# Membaca dataset Parquet dengan skema partisi Hive
skema_partisi = pa.schema([
    ("borough_naik", pa.string())
])

dataset = ds.dataset(
    TRIPS_BERSIH,
    format="parquet",
    partitioning=ds.partitioning(
        skema_partisi,
        flavor="hive"
    )
)

df = dataset.to_table().to_pandas()

# Mengubah nama kolom agar lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar"
}

df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)

# Membuat sampel sebanyak maksimal 300.000 baris
sample = df.sample(
    n=min(300_000, len(df)),
    random_state=42
).reset_index(drop=True)

sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

print("Jumlah data:", f"{len(df):,}")
print("Jumlah sampel:", f"{len(sample):,}")
print("Jumlah kolom:", df.shape[1])

display(sample.head())


Jumlah data: 5,973,820
Jumlah sampel: 300,000
Jumlah kolom: 18


,waktu_jemput,tpep_dropoff_datetime,passenger_count,jarak_mil,zona_jemput,zona_tujuan,payment_type,fare_amount,tip,total_bayar,durasi_menit,jam,metode_bayar,jam_mulai,suhu_c,hujan_mm,tanggal,wilayah_jemput,id_perjalanan
0,2023-01-29 09:30:32,2023-01-29 09:36:22,1.0,0.88,48,246,1,7.9,2.38,14.28,5.833333,9,Kartu kredit,2023-01-29 09:00:00,5.3,0.0,2023-01-29,Manhattan,1
1,2023-01-14 23:51:48,2023-01-15 00:01:10,1.0,1.49,158,144,1,10.7,3.14,18.84,9.366667,23,Kartu kredit,2023-01-14 23:00:00,-1.2,0.0,2023-01-14,Manhattan,2
2,2023-01-18 21:42:41,2023-01-18 22:26:51,1.0,18.45,132,68,1,70.0,20.14,101.94,44.166667,21,Kartu kredit,2023-01-18 21:00:00,4.7,0.0,2023-01-18,Queens,3
3,2023-01-10 11:10:09,2023-01-10 11:28:38,1.0,1.40,68,100,1,16.3,4.06,24.36,18.483333,11,Kartu kredit,2023-01-10 11:00:00,5.2,0.0,2023-01-10,Manhattan,4
4,2023-01-09 20:09:15,2023-01-09 20:16:53,1.0,1.80,142,236,1,10.7,3.10,18.80,7.633333,20,Kartu kredit,2023-01-09 20:00:00,1.2,0.0,2023-01-09,Manhattan,5


In [16]:

import time
import json
import random
import os
import sqlite3

# Fungsi untuk mengukur waktu eksekusi
catatan = []

def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0

    catatan.append({
        "langkah": label,
        "detik": round(detik, 4)
    })

    print(f"[{label}] {detik:.4f} s")
    return hasil

print("Fungsi ukur() berhasil dibuat.")


Fungsi ukur() berhasil dibuat.


## K-2. Tabel Biasa vs Key-value

Tahap ini bertujuan untuk membandingkan dua model penyimpanan data, yaitu relasional menggunakan SQLite dan key-value menggunakan Dictionary Python.

Kedua model menggunakan data perjalanan yang sama. Pengujian dilakukan melalui dua pertanyaan, yaitu mencari 200 perjalanan berdasarkan ID dan menghitung rata-rata total pembayaran berdasarkan wilayah penjemputan.

Hasil pengujian digunakan untuk membandingkan waktu pencarian dan agregasi pada kedua model penyimpanan.

In [17]:

# Kolom yang digunakan dalam percobaan
KOLOM = [
    "id_perjalanan", "waktu_jemput", "zona_jemput",
    "zona_tujuan", "wilayah_jemput", "jarak_mil",
    "total_bayar", "metode_bayar"
]

data = sample[KOLOM].copy()

# Model 1 - Relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")

con = sqlite3.connect("uji.db")

ukur(
    "relasional: muat tabel",
    lambda: data.to_sql("perjalanan", con, index=False)
)

# Model 2 - Key-value: kamus dengan kunci perjalanan:<id>
def bangun_kv():
    return {
        f"perjalanan:{r['id_perjalanan']}": json.dumps(r)
        for r in data.to_dict("records")
    }

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

# Pertanyaan A: ambil SATU perjalanan berdasarkan ID
# Menggunakan 200 ID acak
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [
        cur.execute(
            "SELECT * FROM perjalanan WHERE id_perjalanan = ?",
            (i,)
        ).fetchone()
        for i in id_uji
    ]

def cari_kv():
    return [
        json.loads(kv[f"perjalanan:{i}"])
        for i in id_uji
    ]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)

# Membuat indeks pada kolom id_perjalanan
con.execute(
    "CREATE INDEX idx_id ON perjalanan(id_perjalanan)"
)

r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)

r3 = ukur("cari 200 perjalanan - key-value", cari_kv)

print(
    "hasil sama?",
    r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"]
)

# Pertanyaan B: hitung rata-rata total bayar per wilayah
# Melibatkan SEMUA data
def hitung_sql():
    return con.execute("""
        SELECT wilayah_jemput,
               COUNT(*),
               ROUND(AVG(total_bayar), 2)
        FROM perjalanan
        GROUP BY wilayah_jemput
        ORDER BY 2 DESC, 1
    """).fetchall()

def hitung_kv():
    total, jumlah = {}, {}

    for teks in kv.values():
        d = json.loads(teks)  # Setiap nilai harus dibuka dahulu
        w = d["wilayah_jemput"]

        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1

    return sorted(
        [(w, jumlah[w], round(total[w] / jumlah[w], 2))
         for w in jumlah],
        key=lambda x: (-x[1], x[0])
    )

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)

print(h1[:3])
print("hasil sama?", h1 == h2)


[relasional: muat tabel] 0.9813 s
[key-value: bangun dict] 2.8101 s
jumlah key: 300000
[cari 200 perjalanan - SQL tanpa indeks] 6.4998 s
[cari 200 perjalanan - SQL dengan indeks] 0.0035 s
[cari 200 perjalanan - key-value] 0.0012 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.1981 s
[hitung per wilayah - key-value (buka semua nilai)] 1.3023 s
[('Manhattan', 266924, 22.7), ('Queens', 27324, 70.41), ('Unknown', 3767, 38.57)]
hasil sama? True


## Hasil K-2

Pengujian menggunakan 300.000 data menunjukkan bahwa SQLite lebih cepat dalam pemuatan data (1,2865 detik) dibandingkan Dictionary key-value (4,5976 detik).

Pada pencarian 200 perjalanan, key-value paling cepat dengan waktu 0,0013 detik, diikuti SQL dengan indeks (0,0036 detik) dan SQL tanpa indeks (6,0145 detik).

Untuk agregasi per wilayah, SQLite lebih cepat (0,2121 detik) dibandingkan key-value (1,3454 detik) karena SQL menggunakan GROUP BY untuk mengelompokkan data secara langsung.

Kedua metode menghasilkan nilai yang sama pada pencarian maupun agregasi. Dengan demikian, key-value lebih efisien untuk pencarian berdasarkan ID, sedangkan SQLite lebih sesuai untuk agregasi data.

## K-3. Document: Bentuk Data yang Fleksibel

Tahap ini bertujuan menyimpan data perjalanan dalam bentuk dokumen JSON menggunakan TinyDB. Data disusun secara bersarang (nested), dengan informasi penjemputan, tujuan, dan biaya dalam satu dokumen.

Field `tip` hanya ditambahkan pada pembayaran kartu kredit. Percobaan ini juga menunjukkan fleksibilitas model document dalam menambahkan field baru tanpa mengubah struktur dokumen lainnya.

In [18]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

# Mengubah setiap baris menjadi dokumen JSON bersarang
def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {
            "zona": int(r.zona_jemput),
            "wilayah": r.wilayah_jemput
        },
        "tujuan": {
            "zona": int(r.zona_tujuan)
        },
        "biaya": {
            "jarak_mil": float(r.jarak_mil),
            "total": float(r.total_bayar),
            "metode_bayar": r.metode_bayar
        }
    }

    # Tip hanya dicatat untuk pembayaran kartu kredit
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)

    return dok

# Membuat database TinyDB di memori
db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")

# Menyimpan 20.000 dokumen
docs = [
    ke_dokumen(r)
    for r in sample.head(20_000).itertuples()
]

tabel.insert_multiple(docs)

print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-29 09:30:32",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 246
  },
  "biaya": {
    "jarak_mil": 0.88,
    "total": 14.28,
    "metode_bayar": "Kartu kredit",
    "tip": 2.38
  }
}


In [19]:
Q = Query()

print(
    "jemput di Manhattan & total > 50:",
    len(tabel.search(
        (Q.jemput.wilayah == "Manhattan") &
        (Q.biaya.total > 50)
    ))
)

print(
    "dokumen yang punya field tip:",
    len(tabel.search(Q.biaya.tip.exists()))
)

print(
    "dokumen yang TIDAK punya tip:",
    len(tabel.search(~Q.biaya.tip.exists()))
)

# Menambahkan field baru bernama cuaca
tabel.insert({
    "id_perjalanan": 999_999,
    "waktu_jemput": "2023-01-15 08:00:00",
    "jemput": {"zona": 161, "wilayah": "Manhattan"},
    "tujuan": {"zona": 236},
    "biaya": {
        "jarak_mil": 1.2,
        "total": 14.5,
        "metode_bayar": "Kartu kredit",
        "tip": 3.0
    },
    "cuaca": {"hujan": True, "suhu_c": -2.0}
})

print(
    "dokumen yang punya field cuaca:",
    len(tabel.search(Q.cuaca.exists()))
)

print("total dokumen sekarang:", len(tabel))

jemput di Manhattan & total > 50: 662
dokumen yang punya field tip: 15957
dokumen yang TIDAK punya tip: 4043
dokumen yang punya field cuaca: 1
total dokumen sekarang: 20001


### Penjelasan Kode

1. `ke_dokumen()` mengubah setiap baris perjalanan menjadi dokumen JSON bersarang.
2. `MemoryStorage` menyimpan database TinyDB sementara di memori.
3. `insert_multiple()` memasukkan 20.000 dokumen sekaligus.
4. `Query()` digunakan untuk mencari dokumen berdasarkan field tertentu.
5. `exists()` memeriksa keberadaan field `tip` dan `cuaca`.
6. Field `cuaca` ditambahkan pada satu dokumen tanpa mengubah struktur dokumen lainnya.

Hasil K-3
Pengujian berhasil menyimpan 20.000 dokumen dalam format JSON menggunakan TinyDB. Hasil pencarian menunjukkan terdapat 662 dokumen dengan wilayah penjemputan Manhattan dan total pembayaran di atas 50.

Sebanyak 15.957 dokumen memiliki field tip, sedangkan 4.043 dokumen tidak memilikinya karena tip hanya dicatat untuk pembayaran kartu kredit.

Penambahan field cuaca berhasil dilakukan tanpa mengubah struktur dokumen lainnya. Terdapat 1 dokumen dengan field tersebut dan jumlah keseluruhan menjadi 20.001 dokumen.

Hasil ini menunjukkan bahwa model document memiliki struktur fleksibel karena setiap dokumen dapat memiliki field yang berbeda tanpa memerlukan perubahan skema tabel.

## K-4. Penyimpanan per Kolom: DuckDB pada Parquet

Tahap ini bertujuan menjalankan query SQL secara langsung pada folder Parquet menggunakan DuckDB tanpa memuat data ke tabel terlebih dahulu. Hasilnya dibandingkan dengan SQLite yang menggunakan penyimpanan berbasis baris untuk melihat perbedaan waktu pemrosesan dan ukuran penyimpanan.

In [20]:

# K-4 — Penyimpanan per Kolom: DuckDB pada Parquet

import duckdb
import os
import sqlite3

# Membaca seluruh file Parquet dari folder partisi
GLOB = f"{TRIPS_BERSIH}/*/*.parquet"

con_d = duckdb.connect()

# Membuat VIEW yang membaca Parquet secara langsung tanpa menyalin data.
# Nama kolom asli TLC diubah menjadi nama yang lebih mudah dibaca.
con_d.sql(f"""
CREATE VIEW perjalanan AS
SELECT
    tpep_pickup_datetime AS waktu_jemput,
    PULocationID AS zona_jemput,
    DOLocationID AS zona_tujuan,
    borough_naik AS wilayah_jemput,
    trip_distance AS jarak_mil,
    total_amount AS total_bayar,
    nama_pembayaran AS metode_bayar
FROM read_parquet('{GLOB}', hive_partitioning=true)
""")

# Menghitung jumlah perjalanan dan rata-rata pembayaran setiap wilayah.
SQL = """
SELECT
    wilayah_jemput,
    COUNT(*) AS jumlah,
    ROUND(AVG(total_bayar), 2) AS rata_bayar
FROM perjalanan
GROUP BY wilayah_jemput
ORDER BY jumlah DESC, wilayah_jemput
"""

hasil_d = ukur(
    "DuckDB - hitung per wilayah (langsung dari Parquet)",
    lambda: con_d.sql(SQL).df()
)

print("Hasil agregasi DuckDB:")
print(hasil_d)

# Membandingkan dengan SQLite yang menyimpan data per baris.
KOLOM_S = [
    "wilayah_jemput",
    "total_bayar",
    "jarak_mil",
    "zona_jemput",
    "zona_tujuan"
]

if os.path.exists("baris.db"):
    os.remove("baris.db")

con_s = sqlite3.connect("baris.db")

# Memasukkan data ke tabel SQLite.
ukur(
    "SQLite - muat seluruh baris ke tabel",
    lambda: df[KOLOM_S].to_sql(
        "perjalanan",
        con_s,
        index=False
    )
)

# Menjalankan agregasi yang sama pada SQLite.
hasil_s = ukur(
    "SQLite - hitung per wilayah",
    lambda: con_s.execute("""
        SELECT
            wilayah_jemput,
            COUNT(*),
            ROUND(AVG(total_bayar), 2)
        FROM perjalanan
        GROUP BY wilayah_jemput
        ORDER BY 2 DESC, 1
    """).fetchall()
)

print("Hasil agregasi SQLite:")
print(hasil_s)

# Membandingkan hasil DuckDB dan SQLite.
# Menyamakan representasi partisi kosong.
hasil_d_normal = [
    (
        "None" if r[0] == "__HIVE_DEFAULT_PARTITION__" else r[0],
        r[1],
        r[2]
    )
    for r in hasil_d.itertuples(index=False, name=None)
]

print("Hasil sama?", hasil_d_normal == hasil_s)

# Menguji pembacaan data untuk satu wilayah tertentu.
SQL2 = """
SELECT
    COUNT(*),
    ROUND(AVG(total_bayar), 2)
FROM perjalanan
WHERE wilayah_jemput = 'Staten Island'
"""

hasil_staten = ukur(
    "DuckDB - hanya wilayah Staten Island",
    lambda: con_d.sql(SQL2).fetchall()
)

print("Hasil Staten Island:", hasil_staten)

# Membandingkan ukuran penyimpanan Parquet dan SQLite.
ukuran_pq = sum(
    os.path.getsize(os.path.join(r, f))
    for r, _, fs in os.walk(TRIPS_BERSIH)
    for f in fs
) / 1024**2

print(
    f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | "
    f"SQLite ({len(KOLOM_S)} kolom): "
    f"{os.path.getsize('baris.db') / 1024**2:.1f} MB"
)

# Menutup koneksi database.
con_s.close()
con_d.close()


[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.5262 s
Hasil agregasi DuckDB:
               wilayah_jemput   jumlah  rata_bayar
0                   Manhattan  5319218       22.72
1                      Queens   540630       70.46
2                     Unknown    75218       39.24
3                    Brooklyn    31448       32.72
4                       Bronx     6148       34.44
5  __HIVE_DEFAULT_PARTITION__      680       74.72
6               Staten Island      422       74.07
7                         EWR       56      106.41
[SQLite - muat seluruh baris ke tabel] 16.4463 s
[SQLite - hitung per wilayah] 3.8507 s
Hasil agregasi SQLite:
[('Manhattan', 5319218, 22.72), ('Queens', 540630, 70.46), ('Unknown', 75218, 39.24), ('Brooklyn', 31448, 32.72), ('Bronx', 6148, 34.44), ('None', 680, 74.72), ('Staten Island', 422, 74.07), ('EWR', 56, 106.41)]
Hasil sama? True
[DuckDB - hanya wilayah Staten Island] 0.0678 s
Hasil Staten Island: [(422, 74.07)]
Parquet (18 kolom): 133.1 MB | 

### Hasil K-4

Pengujian dilakukan menggunakan 5.973.820 baris data untuk membandingkan penyimpanan kolom menggunakan Parquet dengan penyimpanan berbasis baris menggunakan SQLite.

DuckDB berhasil melakukan agregasi langsung dari file Parquet dalam waktu 1,1634 detik. Sementara itu, SQLite membutuhkan 15,2073 detik untuk memuat data dan 3,9146 detik untuk melakukan agregasi.

Hasil agregasi kedua metode menunjukkan nilai yang sama, dibuktikan dengan output `Hasil sama? True`. Pengujian khusus wilayah Staten Island juga berhasil dilakukan dengan hasil 422 perjalanan dan rata-rata pembayaran 74,07.

Dari sisi penyimpanan, file Parquet berukuran 133,1 MB, sedangkan SQLite berukuran 230,4 MB. Hasil ini menunjukkan bahwa DuckDB dan Parquet lebih efisien untuk analisis data secara langsung karena mampu melakukan agregasi tanpa harus memuat seluruh data ke dalam tabel terlebih dahulu.

## K-5 — Graph: Pertanyaan tentang Keterhubungan

In [21]:



import duckdb
import networkx as nx
import time

# Membuka kembali koneksi DuckDB karena koneksi K-4 sudah ditutup.
GLOB = f"{TRIPS_BERSIH}/*/*.parquet"

con_d = duckdb.connect()

# Membuat kembali VIEW perjalanan.
con_d.sql(f"""
CREATE VIEW perjalanan AS
SELECT
    tpep_pickup_datetime AS waktu_jemput,
    PULocationID AS zona_jemput,
    DOLocationID AS zona_tujuan,
    borough_naik AS wilayah_jemput,
    trip_distance AS jarak_mil,
    total_amount AS total_bayar,
    nama_pembayaran AS metode_bayar
FROM read_parquet('{GLOB}', hive_partitioning=true)
""")

# Menghitung waktu eksekusi setiap proses.
# Menggunakan fungsi ukur() yang sudah dibuat pada bagian sebelumnya.

# Satu baris = satu arus perjalanan dari zona asal ke zona tujuan.
arus = ukur(
    "hitung arus antar zona",
    lambda: con_d.sql("""
        SELECT
            zona_jemput AS asal,
            zona_tujuan AS tujuan,
            COUNT(*) AS jumlah
        FROM perjalanan
        GROUP BY 1, 2
    """).df()
)

# Menghapus data yang memiliki zona asal atau tujuan kosong.
arus = arus.dropna(subset=["asal", "tujuan"]).copy()

# Membuat graph berarah.
# Node = zona, edge = arus perjalanan, bobot = jumlah perjalanan.
G = nx.DiGraph()

for r in arus.itertuples(index=False):
    G.add_edge(
        int(r.asal),
        int(r.tujuan),
        jumlah=int(r.jumlah)
    )

print(
    "node (zona):", G.number_of_nodes(),
    "| edge (arus):", G.number_of_edges()
)

# Menampilkan lima arus perjalanan terbesar.
top_graf = sorted(
    G.edges(data=True),
    key=lambda e: (-e[2]["jumlah"], e[0], e[1])
)[:5]

print(
    "graph:",
    [(a, b, d["jumlah"]) for a, b, d in top_graf]
)

# Membandingkan lima arus terbesar menggunakan SQL.
top_sql = arus.sort_values(
    ["jumlah", "asal", "tujuan"],
    ascending=[False, True, True]
).head(5)

print(
    "SQL :",
    list(top_sql.itertuples(index=False, name=None))
)

# Menampilkan lima zona yang paling banyak menjadi tujuan.
populer = sorted(
    G.in_degree(weight="jumlah"),
    key=lambda kv: (-kv[1], kv[0])
)[:5]

print("zona tujuan terpopuler:", populer)

# Mengabaikan arus yang memiliki kurang dari 50 perjalanan.
MIN = 50

# Menggunakan zona tujuan terpopuler sebagai zona awal.
zona_awal = populer[0][0]

# Pertanyaan: zona mana yang bisa dicapai dalam tepat dua langkah?

# Cara 1: menggunakan Graph NetworkX.
def dua_langkah_graph():
    hasil = set()

    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue

        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)

    return hasil

# Cara 2: menggunakan SQL self-join.
con_d.register("arus", arus)

def dua_langkah_sql():
    return set(
        con_d.sql(f"""
            SELECT DISTINCT a2.tujuan
            FROM arus a1
            JOIN arus a2
                ON a1.tujuan = a2.asal
            WHERE a1.asal = {zona_awal}
              AND a1.jumlah >= {MIN}
              AND a2.jumlah >= {MIN}
              AND a2.tujuan <> {zona_awal}
        """).df()["tujuan"]
    )

# Menjalankan dan membandingkan kedua metode.
hg = ukur(
    "2 langkah - graph (NetworkX)",
    dua_langkah_graph
)

hs = ukur(
    "2 langkah - SQL self-join (DuckDB)",
    dua_langkah_sql
)

print(
    "zona awal:", zona_awal,
    "| terjangkau (graph):", len(hg),
    "| (SQL):", len(hs),
    "| sama?", hg == hs
)

# Membuat graph baru dengan hanya arus yang memiliki minimal 50 perjalanan.
H = nx.DiGraph([
    (a, b)
    for a, b, d in G.edges(data=True)
    if d["jumlah"] >= MIN
])

# Menghitung zona yang dapat dijangkau dalam maksimal k langkah.
for k in (1, 2, 3, 4):
    n = len(
        nx.single_source_shortest_path_length(
            H,
            zona_awal,
            cutoff=k
        )
    ) - 1

    print(f"dalam <= {k} langkah: {n} zona")

# Menutup koneksi DuckDB.
con_d.close()


[hitung arus antar zona] 0.7152 s
node (zona): 262 | edge (arus): 21583
graph: [(237, 236, 44130), (236, 237, 37620), (236, 236, 28220), (237, 237, 27506), (264, 264, 27124)]
SQL : [(237, 236, 44130), (236, 237, 37620), (236, 236, 28220), (237, 237, 27506), (264, 264, 27124)]
zona tujuan terpopuler: [(236, 288418), (237, 260738), (161, 227982), (230, 175412), (170, 174220)]
[2 langkah - graph (NetworkX)] 0.0074 s
[2 langkah - SQL self-join (DuckDB)] 0.0074 s
zona awal: 236 | terjangkau (graph): 226 | (SQL): 226 | sama? True
dalam <= 1 langkah: 91 zona
dalam <= 2 langkah: 226 zona
dalam <= 3 langkah: 226 zona
dalam <= 4 langkah: 226 zona


Penjelasan Kode
1. Membuat arus perjalanan: SQL mengelompokkan perjalanan berdasarkan zona asal dan tujuan serta menghitung jumlah perjalanan.
2. Membuat graph: NetworkX membentuk graph berarah. Setiap zona menjadi node, sedangkan hubungan antar zona menjadi edge dengan bobot jumlah perjalanan.
3. Arus terbesar: Menampilkan lima hubungan perjalanan dengan jumlah perjalanan terbanyak dan membandingkannya dengan hasil SQL.
4. Zona tujuan terpopuler: Menghitung lima zona yang paling sering menjadi tujuan menggunakan in_degree.
5. Pencarian dua langkah: Membandingkan zona yang dapat dicapai melalui dua langkah menggunakan NetworkX dan SQL self-join.
6. Jangkauan graph: Menghitung jumlah zona yang dapat dijangkau dalam maksimal 1, 2, 3, dan 4 langkah dengan batas minimal 50 perjalanan.

### Hasil K-5

Pengujian graph menghasilkan 262 node dan 21.583 edge yang merepresentasikan hubungan perjalanan antar zona. Lima arus perjalanan terbesar berhasil diidentifikasi, dengan arus tertinggi dari zona 237 menuju zona 236 sebanyak 44.130 perjalanan.

Zona tujuan terpopuler adalah zona 236 dengan jumlah 288.418 perjalanan masuk. Pengujian pencarian dua langkah menggunakan NetworkX dan SQL self-join menghasilkan nilai yang sama, yaitu 226 zona, dengan waktu eksekusi masing-masing 0,0077 detik dan 0,0198 detik.

Berdasarkan pengujian jangkauan, terdapat 91 zona yang dapat dicapai dalam satu langkah dan 226 zona dalam maksimal dua langkah. Jumlah tersebut tidak bertambah pada langkah ketiga dan keempat.

Hasil ini menunjukkan bahwa Graph dapat digunakan untuk menganalisis keterhubungan antar zona perjalanan. NetworkX dan SQL memberikan hasil yang konsisten, sedangkan NetworkX memiliki waktu eksekusi lebih cepat pada pengujian pencarian dua langkah ini.

## K-6 — Indeks Spasial

In [22]:

# K-6 — Indeks Spasial

import numpy as np
from shapely import STRtree, box, points
import time

# Fungsi pengukur waktu jika belum tersedia.
if "ukur" not in globals():
    catatan = []

    def ukur(label, fungsi):
        t0 = time.perf_counter()
        hasil = fungsi()
        detik = time.perf_counter() - t0

        catatan.append({
            "langkah": label,
            "detik": round(detik, 4)
        })

        print(f"[{label}] {detik:.4f} s")
        return hasil

# CATATAN:
# Data TLC 2023 hanya memiliki ID zona, bukan koordinat.
# Oleh karena itu, titik koordinat berikut merupakan data sintetis.

rng = np.random.default_rng(42)

# Membuat 200.000 titik koordinat sintetis.
N = 200_000

bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)

titik = points(bujur, lintang)

# Membuat 1.000 kotak pencarian berukuran sekitar 1 km x 1 km.
M = 1000

cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)

kotak = [
    box(x - 0.005, y - 0.005, x + 0.005, y + 0.005)
    for x, y in zip(cx, cy)
]

# CARA 1: Loop Python.
# Memeriksa titik satu per satu untuk 5 kotak pertama.
def cara_loop():
    return [
        sum(1 for p in titik if k.contains(p))
        for k in kotak[:5]
    ]

# CARA 2: NumPy tanpa indeks spasial.
# Membandingkan seluruh koordinat titik dengan setiap kotak.
def cara_numpy():
    hasil = []

    for x, y in zip(cx, cy):
        m = (
            (bujur >= x - 0.005) &
            (bujur <= x + 0.005) &
            (lintang >= y - 0.005) &
            (lintang <= y + 0.005)
        )

        hasil.append(int(m.sum()))

    return hasil

# Menjalankan metode pertama dan kedua.
h_loop = ukur(
    "cara 1 - loop Python, 5 kotak",
    cara_loop
)

h_np = ukur(
    "cara 2 - NumPy, 1000 kotak",
    cara_numpy
)

# CARA 3: STRtree sebagai indeks spasial.
# Membangun indeks dari seluruh titik.
pohon = ukur(
    "bangun indeks STRtree",
    lambda: STRtree(titik)
)

# Mencari titik yang berada di dalam setiap kotak.
h_idx = ukur(
    "cara 3 - STRtree, 1000 kotak",
    lambda: [
        len(pohon.query(k, predicate="contains"))
        for k in kotak
    ]
)

# Membandingkan hasil ketiga metode.
print(
    "5 kotak pertama - loop:",
    h_loop,
    "| STRtree:",
    h_idx[:5]
)

print(
    "hasil NumPy dan STRtree sama?",
    h_np == h_idx,
    "| loop dan STRtree sama (5 pertama)?",
    h_loop == h_idx[:5]
)

# Menghitung total titik yang ditemukan dalam 1.000 kotak.
print(
    "total titik ditemukan (1000 kotak):",
    sum(h_idx)
)


[cara 1 - loop Python, 5 kotak] 0.3422 s
[cara 2 - NumPy, 1000 kotak] 0.7070 s
[bangun indeks STRtree] 0.0983 s
[cara 3 - STRtree, 1000 kotak] 0.3026 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


### Hasil K-6

Pengujian dilakukan menggunakan 200.000 titik koordinat dan 1.000 kotak wilayah untuk membandingkan tiga metode pencarian spasial.

Metode loop Python membutuhkan waktu 2,3231 detik untuk memproses 5 kotak pertama, sedangkan NumPy membutuhkan 1,5460 detik untuk memproses seluruh 1.000 kotak. Pembuatan indeks STRtree membutuhkan 0,0841 detik, kemudian pencarian menggunakan STRtree membutuhkan 0,4100 detik untuk seluruh 1.000 kotak.

Hasil pencarian pada lima kotak pertama menggunakan loop Python dan STRtree menunjukkan nilai yang sama. Selain itu, hasil NumPy dan STRtree juga identik, yang dibuktikan dengan output `True` pada kedua validasi. Secara keseluruhan, terdapat 417.545 titik yang ditemukan dalam 1.000 kotak.

Hasil ini menunjukkan bahwa STRtree dapat mempercepat pencarian spasial dengan memanfaatkan indeks spasial. Meskipun membutuhkan waktu untuk membangun indeks, STRtree lebih efisien untuk pencarian berulang pada banyak wilayah dibandingkan pemeriksaan satu per satu menggunakan loop Python.

## K-7. DELTA LAKE: VERSI, TIME TRAVEL, DAN SCHEMA ENFORCEMEN

In [23]:

# ============================================================
# K-7. DELTA LAKE: VERSI, TIME TRAVEL, DAN SCHEMA ENFORCEMENT
# ============================================================

%pip -q install deltalake pyarrow

import os
import shutil
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
from deltalake import DeltaTable, write_deltalake

# ------------------------------------------------------------
# 1. PERSIAPAN DATA
# ------------------------------------------------------------

TRIPS_BERSIH = "/content/drive/MyDrive/BigData/Praktikum4/trips_bersih"

# Gunakan penyimpanan lokal Colab, bukan Google Drive
PATH_DELTA = "/content/trips_delta"

# Membaca dataset jika df belum tersedia
if "df" not in globals():

    skema_partisi = pa.schema([
        ("borough_naik", pa.string())
    ])

    dataset = ds.dataset(
        TRIPS_BERSIH,
        format="parquet",
        partitioning=ds.partitioning(
            skema_partisi,
            flavor="hive"
        )
    )

    df = dataset.to_table().to_pandas()

    NAMA_BARU = {
        "tpep_pickup_datetime": "waktu_jemput",
        "PULocationID": "zona_jemput",
        "DOLocationID": "zona_tujuan",
        "borough_naik": "wilayah_jemput",
        "trip_distance": "jarak_mil",
        "total_amount": "total_bayar",
        "tip_amount": "tip",
        "nama_pembayaran": "metode_bayar"
    }

    df = df.rename(columns=NAMA_BARU)

KOLOM_DELTA = [
    "waktu_jemput",
    "zona_jemput",
    "zona_tujuan",
    "wilayah_jemput",
    "jarak_mil",
    "total_bayar",
    "metode_bayar"
]

data_delta = df[KOLOM_DELTA].sample(
    n=min(200_000, len(df)),
    random_state=42
).reset_index(drop=True)

data_delta["waktu_jemput"] = data_delta["waktu_jemput"].astype(str)
data_delta["wilayah_jemput"] = data_delta["wilayah_jemput"].astype(str)
data_delta["metode_bayar"] = data_delta["metode_bayar"].astype(str)

# Hapus tabel Delta sebelumnya jika ada
if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

# ------------------------------------------------------------
# 2. VERSI 0: MENULIS 100.000 BARIS
# ------------------------------------------------------------

write_deltalake(
    PATH_DELTA,
    data_delta.iloc[:100_000],
    mode="overwrite"
)

dt = DeltaTable(PATH_DELTA)

print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# ------------------------------------------------------------
# 3. VERSI 1: MENAMBAHKAN 50.000 BARIS
# ------------------------------------------------------------

write_deltalake(
    PATH_DELTA,
    data_delta.iloc[100_000:150_000],
    mode="append"
)

dt = DeltaTable(PATH_DELTA)

print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# ------------------------------------------------------------
# 4. VERSI 2: UPDATE DATA
# ------------------------------------------------------------

sekarang = data_delta.iloc[:150_000]

n_koreksi = int(
    (
        (sekarang["wilayah_jemput"] == "Bronx") &
        (sekarang["metode_bayar"] == "Tunai")
    ).sum()
)

print("baris yang akan dikoreksi:", n_koreksi)

dt.update(
    predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
    updates={"total_bayar": "total_bayar + 1.0"}
)

print("versi:", DeltaTable(PATH_DELTA).version())

# ------------------------------------------------------------
# 5. TIME TRAVEL: MEMBACA VERSI SEBELUMNYA
# ------------------------------------------------------------

v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()

print(
    "baris v0:", len(v0),
    "| v1:", len(v1),
    "| terbaru:", len(terbaru)
)

selisih = (
    terbaru["total_bayar"].sum() -
    v1["total_bayar"].sum()
)

print(
    "selisih total_bayar (terbaru - v1):",
    round(selisih, 2),
    "| jumlah baris dikoreksi:",
    n_koreksi
)

# ------------------------------------------------------------
# 6. RIWAYAT PERUBAHAN TABEL
# ------------------------------------------------------------

print("\nRiwayat perubahan tabel:")

for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

# Menampilkan file log transaksi
print("\nFile log transaksi:")

print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

# ------------------------------------------------------------
# 7. SCHEMA ENFORCEMENT
# ------------------------------------------------------------

tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1

try:
    write_deltalake(
        PATH_DELTA,
        tambahan,
        mode="append"
    )

    print("append kolom asing: BERHASIL (tidak diharapkan)")

except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)

print(
    "versi setelah percobaan gagal:",
    DeltaTable(PATH_DELTA).version()
)

print("\nPENGUJIAN K-7 SELESAI")


versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 27
versi: 2
baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 27.0 | jumlah baris dikoreksi: 27

Riwayat perubahan tabel:
2 UPDATE
1 WRITE
0 WRITE

File log transaksi:
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2

PENGUJIAN K-7 SELESAI


### Hasil K-7

Pengujian berhasil menghasilkan tiga versi tabel Delta Lake, yaitu versi 0 (100.000 baris), versi 1 (150.000 baris), dan versi 2 setelah proses UPDATE.

Sebanyak 27 baris berhasil dikoreksi dengan penambahan pembayaran sebesar 1,0. Fitur Time Travel berhasil membaca versi sebelumnya, sedangkan Schema Enforcement berhasil menolak penambahan kolom asing (`SchemaMismatchError`).

Hasil ini membuktikan bahwa Delta Lake mendukung pengelolaan versi, pemulihan data versi sebelumnya, dan konsistensi skema.

## K-8. SMALL FILE PROBLEM PADA TABEL DELTA

In [24]:
import os
import time
import glob
from deltalake import DeltaTable, write_deltalake

# Pastikan tabel Delta dan data tersedia
assert os.path.exists(PATH_DELTA), "Tabel Delta K-7 tidak ditemukan."
assert "data_delta" in globals(), "Data data_delta tidak tersedia."

# ------------------------------------------------------------
# 1. KONDISI SEBELUM APPEND
# ------------------------------------------------------------

dt = DeltaTable(PATH_DELTA)

print(
    "file data aktif sebelum:",
    len(dt.file_uris()),
    "| versi:",
    dt.version()
)

# ------------------------------------------------------------
# 2. APPEND 20 KALI DATA KECIL
# ------------------------------------------------------------

for i in range(20):
    potongan = data_delta.iloc[
        150_000 + i * 500: 150_000 + (i + 1) * 500
    ]

    write_deltalake(
        PATH_DELTA,
        potongan,
        mode="append"
    )

dt = DeltaTable(PATH_DELTA)

print(
    "file data aktif setelah 20 append kecil:",
    len(dt.file_uris()),
    "| versi:",
    dt.version(),
    "| baris:",
    len(dt.to_pandas())
)

# ------------------------------------------------------------
# 3. MENGUKUR WAKTU BACA SEBELUM COMPACTION
# ------------------------------------------------------------

t0 = time.perf_counter()
_ = DeltaTable(PATH_DELTA).to_pandas()
t_sebelum = time.perf_counter() - t0

# ------------------------------------------------------------
# 4. COMPACTION
# ------------------------------------------------------------

hasil = dt.optimize.compact()

print(
    "hasil compact:",
    {
        k: hasil[k]
        for k in ("numFilesAdded", "numFilesRemoved")
    }
)

dt = DeltaTable(PATH_DELTA)

# ------------------------------------------------------------
# 5. MENGUKUR WAKTU BACA SESUDAH COMPACTION
# ------------------------------------------------------------

t0 = time.perf_counter()
_ = DeltaTable(PATH_DELTA).to_pandas()
t_sesudah = time.perf_counter() - t0

print(
    "file data aktif setelah compact:",
    len(dt.file_uris()),
    "| versi:",
    dt.version(),
    "| baris:",
    len(dt.to_pandas())
)

print(
    f"waktu baca sebelum compact: {t_sebelum:.4f} s "
    f"| sesudah: {t_sesudah:.4f} s"
)

# ------------------------------------------------------------
# 6. MEMERIKSA FILE DAN VACUUM (DRY RUN)
# ------------------------------------------------------------

print(
    "file parquet di disk:",
    len(glob.glob(f"{PATH_DELTA}/**/*.parquet", recursive=True))
)

kandidat = dt.vacuum(
    retention_hours=0,
    enforce_retention_duration=False,
    dry_run=True
)

print(
    "file yang akan dihapus VACUUM (dry run):",
    len(kandidat)
)

print("\nPENGUJIAN K-8 SELESAI")


file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.2368 s | sesudah: 0.1190 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23

PENGUJIAN K-8 SELESAI


### Hasil K-8

Pengujian menghasilkan 21 file aktif setelah 20 kali append data kecil, dengan total 160.000 baris. Setelah compaction, 21 file berhasil digabung menjadi 1 file tanpa mengubah jumlah baris.

Waktu baca berkurang dari 0,1775 detik menjadi 0,1447 detik. Sebanyak 24 file Parquet masih tersimpan di disk, dan 23 file teridentifikasi sebagai kandidat penghapusan melalui VACUUM (*dry run*).

Hasil ini menunjukkan bahwa compaction mengurangi jumlah file aktif dan mempercepat proses pembacaan data.

## K-9. MERANGKUM DAN MEMILIH

In [25]:
import os
import pandas as pd

# Pastikan folder penyimpanan tersedia
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
os.makedirs(DIR_SIMPAN, exist_ok=True)

# Simpan seluruh catatan benchmark
if "catatan" not in globals():
    catatan = []

# Tambahkan waktu baca K-8 jika belum tercatat
if "t_sebelum" in globals():
    catatan.append({
        "langkah": "K-8 baca sebelum compaction",
        "detik": round(t_sebelum, 4)
    })

if "t_sesudah" in globals():
    catatan.append({
        "langkah": "K-8 baca sesudah compaction",
        "detik": round(t_sesudah, 4)
    })

rekap = pd.DataFrame(catatan)

PATH_CSV = f"{DIR_SIMPAN}/benchmark_nosql.csv"
rekap.to_csv(PATH_CSV, index=False)

print("Benchmark berhasil disimpan:", PATH_CSV)
print("\nRekap benchmark:")
print(rekap.to_string(index=False))


Benchmark berhasil disimpan: /content/drive/MyDrive/BigData/Praktikum4/benchmark_nosql.csv

Rekap benchmark:
                                            langkah   detik
                             relasional: muat tabel  0.9813
                             key-value: bangun dict  2.8101
             cari 200 perjalanan - SQL tanpa indeks  6.4998
            cari 200 perjalanan - SQL dengan indeks  0.0035
                    cari 200 perjalanan - key-value  0.0012
                  hitung per wilayah - SQL GROUP BY  0.1981
  hitung per wilayah - key-value (buka semua nilai)  1.3023
DuckDB - hitung per wilayah (langsung dari Parquet)  0.5262
               SQLite - muat seluruh baris ke tabel 16.4463
                        SQLite - hitung per wilayah  3.8507
               DuckDB - hanya wilayah Staten Island  0.0678
                             hitung arus antar zona  0.7152
                       2 langkah - graph (NetworkX)  0.0074
                 2 langkah - SQL self-join (DuckDB)

### Hasil K-9

| Kebutuhan | Cara unggul | Angka Anda |
|---|---|---|
| Pencarian berdasarkan ID | Key-value | 0,0013 detik |
| Agregasi banyak baris | DuckDB + Parquet | 1,1634 detik |
| Data dengan field opsional | Document | Kualitatif |
| Keterhubungan antar-zona | Graph (NetworkX) | 0,0077 detik |
| Pencarian lokasi dalam wilayah | STRtree | 0,4100 detik |
| Perubahan data yang aman | Delta Lake | 27 baris berhasil dikoreksi |

# P. Studi Kasus: Rancangan Penyimpanan untuk Aplikasi Taksi

## 1. Pendahuluan

Aplikasi taksi membutuhkan beberapa jenis penyimpanan karena setiap fitur memiliki kebutuhan akses dan pengolahan data yang berbeda. Berdasarkan hasil Praktikum 4, rancangan ini menggunakan kombinasi Key-Value, Parquet dengan DuckDB, Graph, indeks spasial STRtree, dan Delta Lake.

## 2. Rancangan Penyimpanan

| Fitur | Model penyimpanan | Teknologi | Cara kerja dan alasan pemilihan |
|---|---|---|---|
| Detail perjalanan berdasarkan ID | Key-Value | Dictionary / Redis | Data disimpan dengan ID perjalanan sebagai key sehingga dapat dicari secara langsung. Pada pengujian K-2, pencarian membutuhkan 0,0013 detik. |
| Dashboard pendapatan per wilayah per jam | Column-oriented | Parquet + DuckDB | Data dikelompokkan berdasarkan wilayah dan waktu menggunakan SQL GROUP BY. DuckDB dapat membaca kolom yang diperlukan secara langsung dari Parquet. |
| Rekomendasi zona tujuan dan jangkauan beberapa langkah | Graph | NetworkX / Graph Database | Zona perjalanan direpresentasikan sebagai node dan hubungan perjalanan sebagai edge. Graph memudahkan pencarian hubungan satu hingga beberapa langkah. |
| Pencarian taksi dalam radius 500 meter | Spatial Index | STRtree | Koordinat lokasi penjemputan disimpan sebagai titik spasial. STRtree digunakan untuk mencari titik yang berada dalam area pencarian. |
| Audit keuangan dan pemulihan data | Versioned Table | Delta Lake | Perubahan data dicatat dalam versi sehingga tabel dapat dibaca kembali menggunakan Time Travel jika terjadi kesalahan koreksi. |

## 3. Diagram Aliran Data

Data perjalanan dari folder `trips_bersih/` diproses dan didistribusikan ke beberapa sistem penyimpanan sesuai kebutuhan fitur.

```mermaid
flowchart TD
    A["trips_bersih/ (Parquet)"]
    B["Proses ETL dan pembersihan data"]
    C["Key-Value Store"]
    D["Parquet + DuckDB"]
    E["Graph Store"]
    F["Spatial Index (STRtree)"]
    G["Delta Lake"]

    H["Detail Perjalanan"]
    I["Dashboard Pendapatan"]
    J["Rekomendasi Rute"]
    K["Pencarian Radius 500 m"]
    L["Audit Keuangan"]

    A --> B
    B --> C
    B --> D
    B --> E
    B --> F
    B --> G

    C --> H
    D --> I
    E --> J
    F --> K
    G --> L
```

## 4. Alasan Pemilihan Arsitektur

Kombinasi beberapa model penyimpanan dipilih karena tidak semua kebutuhan aplikasi dapat ditangani secara efisien oleh satu jenis database.

- **Key-Value** digunakan untuk pencarian detail perjalanan berdasarkan ID dengan cepat.
- **Parquet dan DuckDB** digunakan untuk analisis pendapatan karena mendukung agregasi data dalam jumlah besar.
- **Graph** digunakan untuk rekomendasi rute karena mampu menelusuri hubungan antar-zona dalam beberapa langkah.
- **STRtree** digunakan untuk pencarian lokasi berdasarkan radius agar pencarian spasial lebih efisien.
- **Delta Lake** digunakan untuk audit keuangan karena menyediakan riwayat perubahan, Time Travel, dan Schema Enforcement.

Perlu diperhatikan bahwa dataset praktikum saat ini menggunakan ID zona dan belum menyediakan koordinat geografis asli untuk setiap perjalanan. Oleh karena itu, implementasi fitur radius 500 meter memerlukan data koordinat penjemputan yang sebenarnya.

## 5. Kesimpulan

Rancangan aplikasi taksi menggunakan lima pendekatan penyimpanan yang disesuaikan dengan kebutuhan masing-masing fitur. Kombinasi tersebut memungkinkan pencarian cepat, analisis pendapatan, rekomendasi rute, pencarian spasial, serta audit dan pemulihan data secara terstruktur.

In [26]:

# ============================================================
# Q. LATIHAN 1-6
# ============================================================

import time
import random
import json
import numpy as np
import duckdb
import networkx as nx
from shapely import STRtree, box, points
from deltalake import DeltaTable

# ============================================================
# LATIHAN 1: PENCARIAN 2.000 ID ACAK
# ============================================================

print("\n========== LATIHAN 1 ==========")

assert "con" in globals() and "kv" in globals(), "Jalankan K-2 terlebih dahulu."

random.seed(42)
id_2000 = random.sample(range(1, len(data) + 1), 2000)

def cari_sql_indeks_2000():
    cur = con.cursor()
    for id_data in id_2000:
        cur.execute(
            "SELECT * FROM perjalanan WHERE id_perjalanan = ?",
            (id_data,)
        ).fetchone()

def cari_kv_2000():
    for id_data in id_2000:
        json.loads(kv[f"perjalanan:{id_data}"])

t0 = time.perf_counter()
cari_sql_indeks_2000()
waktu_sql = time.perf_counter() - t0

t0 = time.perf_counter()
cari_kv_2000()
waktu_kv = time.perf_counter() - t0

print("Jumlah pencarian:", len(id_2000))
print(f"SQL berindeks: {waktu_sql / 2000 * 1_000_000:.2f} mikrodetik/pencarian")
print(f"Key-value: {waktu_kv / 2000 * 1_000_000:.2f} mikrodetik/pencarian")


# ============================================================
# LATIHAN 2: DOCUMENT VS SQL
# ============================================================

print("\n========== LATIHAN 2 ==========")

assert "tabel" in globals(), "Jalankan K-3 terlebih dahulu."

ZONA_TUJUAN = 236

from tinydb import Query
Q = Query()

jumlah_document = len(
    tabel.search(
        (Q.tujuan.zona == ZONA_TUJUAN) &
        (Q.biaya.metode_bayar == "Tunai")
    )
)

# Dokumen TinyDB berasal dari 20.000 data pertama.
# Batasi SQL pada ID yang sama agar perbandingannya adil.
jumlah_sql = con.execute("""
    SELECT COUNT(*)
    FROM perjalanan
    WHERE zona_tujuan = ?
      AND metode_bayar = 'Tunai'
      AND id_perjalanan <= 20000
""", (ZONA_TUJUAN,)).fetchone()[0]

print("Zona tujuan:", ZONA_TUJUAN)
print("Jumlah document:", jumlah_document)
print("Jumlah SQL:", jumlah_sql)
print("Hasil sama?", jumlah_document == jumlah_sql)


# ============================================================
# LATIHAN 3: SUM SATU KOLOM VS BEBERAPA KOLOM
# ============================================================

print("\n========== LATIHAN 3 ==========")

TRIPS_BERSIH = "/content/drive/MyDrive/BigData/Praktikum4/trips_bersih"
GLOB = f"{TRIPS_BERSIH}/*/*.parquet"

con_q = duckdb.connect()

con_q.execute(f"""
CREATE VIEW perjalanan AS
SELECT
    total_amount AS total_bayar,
    trip_distance AS jarak_mil,
    PULocationID AS zona_jemput,
    DOLocationID AS zona_tujuan
FROM read_parquet('{GLOB}', hive_partitioning=true)
""")

def sum_satu_kolom():
    return con_q.execute("""
        SELECT SUM(total_bayar)
        FROM perjalanan
    """).fetchone()

def sum_beberapa_kolom():
    return con_q.execute("""
        SELECT
            SUM(total_bayar),
            SUM(jarak_mil),
            SUM(zona_jemput),
            SUM(zona_tujuan)
        FROM perjalanan
    """).fetchone()

# Pengukuran rata-rata dari 3 pengulangan
def ukur_rata_rata(fungsi, ulang=3):
    waktu = []
    hasil = None

    for _ in range(ulang):
        t0 = time.perf_counter()
        hasil = fungsi()
        waktu.append(time.perf_counter() - t0)

    return hasil, sum(waktu) / len(waktu)

hasil_satu, waktu_satu = ukur_rata_rata(sum_satu_kolom)
hasil_banyak, waktu_banyak = ukur_rata_rata(sum_beberapa_kolom)

print("SUM satu kolom:", hasil_satu)
print("SUM beberapa kolom:", hasil_banyak)
print(f"Waktu SUM satu kolom: {waktu_satu:.4f} detik")
print(f"Waktu SUM beberapa kolom: {waktu_banyak:.4f} detik")

con_q.close()


# ============================================================
# LATIHAN 4: OUT-DEGREE GRAPH VS SQL
# ============================================================

print("\n========== LATIHAN 4 ==========")

assert "G" in globals(), "Jalankan K-5 terlebih dahulu."
assert "arus" in globals(), "Data arus K-5 tidak tersedia."

# Graph: out-degree = jumlah tujuan berbeda dari setiap zona asal
top_graph = sorted(
    G.out_degree(),
    key=lambda x: (-x[1], x[0])
)[:5]

# SQL: COUNT DISTINCT tujuan
con_graph = duckdb.connect()
con_graph.register("arus_q", arus)

top_sql = con_graph.execute("""
    SELECT
        CAST(asal AS INTEGER) AS asal,
        COUNT(DISTINCT tujuan) AS jumlah_tujuan
    FROM arus_q
    GROUP BY asal
    ORDER BY jumlah_tujuan DESC, asal
    LIMIT 5
""").fetchall()

print("Lima zona berdasarkan Graph:", top_graph)
print("Lima zona berdasarkan SQL:", top_sql)

print(
    "Hasil sama?",
    top_graph == [(a, b) for a, b in top_sql]
)

con_graph.close()


# ============================================================
# LATIHAN 5: STRTREE DENGAN UKURAN KOTAK BERBEDA
# ============================================================

print("\n========== LATIHAN 5 ==========")

rng = np.random.default_rng(42)

N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)

titik = points(bujur, lintang)

M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)

# Bangun indeks satu kali
t0 = time.perf_counter()
pohon = STRtree(titik)
waktu_bangun = time.perf_counter() - t0

print(f"Waktu membangun STRtree: {waktu_bangun:.4f} detik")

for sisi in [0.02, 0.001]:

    setengah = sisi / 2

    kotak_uji = [
        box(x - setengah, y - setengah, x + setengah, y + setengah)
        for x, y in zip(cx, cy)
    ]

    # NumPy tanpa indeks
    t0 = time.perf_counter()

    hasil_np = [
        int((
            (bujur >= x - setengah) &
            (bujur <= x + setengah) &
            (lintang >= y - setengah) &
            (lintang <= y + setengah)
        ).sum())
        for x, y in zip(cx, cy)
    ]

    waktu_np = time.perf_counter() - t0

    # STRtree
    t0 = time.perf_counter()

    hasil_idx = [
        len(pohon.query(k, predicate="contains"))
        for k in kotak_uji
    ]

    waktu_idx = time.perf_counter() - t0

    print(f"\nUkuran sisi kotak: {sisi} derajat")
    print(f"NumPy: {waktu_np:.4f} detik")
    print(f"STRtree: {waktu_idx:.4f} detik")
    print(f"Perbandingan NumPy/STRtree: {waktu_np / waktu_idx:.2f} kali")
    print("Hasil sama?", hasil_np == hasil_idx)


# ============================================================
# LATIHAN 6: TIME TRAVEL VS RESTORE
# JALANKAN PALING AKHIR KARENA MENGUBAH TABEL AKTIF
# ============================================================

print("\n========== LATIHAN 6 ==========")

dt = DeltaTable(PATH_DELTA)

print(
    "versi sebelum restore:",
    dt.version(),
    "| baris:",
    len(dt.to_pandas())
)

# Mengembalikan tabel aktif ke keadaan versi 1
dt.restore(1)

dt = DeltaTable(PATH_DELTA)

print(
    "versi setelah restore:",
    dt.version(),
    "| baris:",
    len(dt.to_pandas())
)

print(
    "Riwayat tiga operasi terakhir:",
    [h.get("operation") for h in dt.history()[:3]]
)

print("\nSELURUH LATIHAN Q SELESAI")



========== LATIHAN 1 ==========
Jumlah pencarian: 2000
SQL berindeks: 28.66 mikrodetik/pencarian
Key-value: 14.75 mikrodetik/pencarian

========== LATIHAN 2 ==========
Zona tujuan: 236
Jumlah document: 129
Jumlah SQL: 129
Hasil sama? True

========== LATIHAN 3 ==========
SUM satu kolom: (163242884.91958073,)
SUM beberapa kolom: (163242884.9195766, 20463779.21999926, 994905510, 982660800)
Waktu SUM satu kolom: 0.6095 detik
Waktu SUM beberapa kolom: 1.3961 detik

========== LATIHAN 4 ==========
Lima zona berdasarkan Graph: [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
Lima zona berdasarkan SQL: [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
Hasil sama? True

========== LATIHAN 5 ==========
Waktu membangun STRtree: 0.1969 detik

Ukuran sisi kotak: 0.02 derajat
NumPy: 1.1103 detik
STRtree: 3.3242 detik
Perbandingan NumPy/STRtree: 0.33 kali
Hasil sama? True

Ukuran sisi kotak: 0.001 derajat
NumPy: 0.7086 detik
STRtree: 0.0217 detik
Perbandingan NumPy/STRtree: 3

## Q. Hasil dan Analisis Latihan

**1. Latihan 1 – Pencarian 2.000 ID**

Pencarian key-value membutuhkan 17,72 mikrodetik per pencarian, sedangkan SQL berindeks membutuhkan 79,43 mikrodetik. Dengan demikian, key-value sekitar 4,48 kali lebih cepat karena pencarian berdasarkan key dilakukan secara langsung.

**2. Latihan 2 – Document vs SQL**

Kedua metode menghasilkan 129 dokumen untuk zona tujuan 236 dengan metode pembayaran Tunai. Hasilnya sama (`True`) karena kedua metode menggunakan data dan kondisi pencarian yang sama.

**3. Latihan 3 – Column Pruning**

SUM satu kolom membutuhkan 0,9454 detik, sedangkan SUM beberapa kolom membutuhkan 1,6954 detik. SUM satu kolom lebih cepat karena hanya membaca kolom yang diperlukan melalui column pruning.

**4. Latihan 4 – Graph vs SQL**

Graph dan SQL menghasilkan lima zona yang sama, yaitu 132, 138, 264, 230, dan 140. Keduanya konsisten karena sama-sama menghitung jumlah zona tujuan berbeda menggunakan out-degree dan COUNT DISTINCT.

**5. Latihan 5 – Indeks Spasial**

Pada ukuran kotak 0,02 derajat, STRtree membutuhkan 0,8712 detik dan NumPy 0,7029 detik, sehingga STRtree lebih lambat. Pada ukuran 0,001 derajat, STRtree membutuhkan 0,0165 detik dan NumPy 0,5572 detik, sehingga STRtree sekitar 33,87 kali lebih cepat. Kotak kecil lebih menguntungkan indeks karena pencarian dapat mempersempit kandidat titik yang diperiksa.

**6. Latihan 6 – Time Travel dan Restore**

Time Travel digunakan untuk membaca data versi lama tanpa mengubah versi aktif, sedangkan Restore mengembalikan kondisi tabel aktif ke versi sebelumnya. Hasil pengujian menunjukkan jumlah baris berubah dari 160.000 menjadi 150.000 dan versi bertambah dari 23 menjadi 24. Restore dicatat sebagai versi baru agar riwayat perubahan tetap tersimpan.

In [27]:
import os

PATH_DELTA = "/content/trips_delta"
LOG_DIR = os.path.join(PATH_DELTA, "_delta_log")

print("Folder Delta ada:", os.path.exists(PATH_DELTA))
print("Folder _delta_log ada:", os.path.exists(LOG_DIR))

if os.path.exists(LOG_DIR):
    files = sorted(os.listdir(LOG_DIR))
    print("Jumlah file:", len(files))

    for f in files:
        print(f)

Folder Delta ada: True
Folder _delta_log ada: True
Jumlah file: 25
00000000000000000000.json
00000000000000000001.json
00000000000000000002.json
00000000000000000003.json
00000000000000000004.json
00000000000000000005.json
00000000000000000006.json
00000000000000000007.json
00000000000000000008.json
00000000000000000009.json
00000000000000000010.json
00000000000000000011.json
00000000000000000012.json
00000000000000000013.json
00000000000000000014.json
00000000000000000015.json
00000000000000000016.json
00000000000000000017.json
00000000000000000018.json
00000000000000000019.json
00000000000000000020.json
00000000000000000021.json
00000000000000000022.json
00000000000000000023.json
00000000000000000024.json


In [28]:
import os

log_dir = "/content/trips_delta/_delta_log"

for i in range(3):
    filename = f"{i:020d}.json"
    path = os.path.join(log_dir, filename)

    print(f"\n===== {filename} =====")
    with open(path, "r") as f:
        print(f.read())


===== 00000000000000000000.json =====
{"commitInfo":{"timestamp":1794653686302,"operation":"WRITE","operationParameters":{"mode":"Overwrite"},"engineInfo":"delta-rs:py-1.6.6","clientVersion":"delta-rs.py-1.6.6","operationMetrics":{"num_added_files":1,"num_removed_files":0,"num_partitions":0,"num_added_rows":100000,"execution_time_ms":67,"num_retries":0}}}
{"protocol":{"minReaderVersion":1,"minWriterVersion":2}}
{"metaData":{"id":"7c53b420-96ec-44b1-b20c-58219ba0aec4","name":null,"description":null,"format":{"provider":"parquet","options":{}},"schemaString":"{\"type\":\"struct\",\"fields\":[{\"name\":\"waktu_jemput\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"zona_jemput\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"zona_tujuan\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"wilayah_jemput\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"jarak_mil\",\"type\":\"double\",\"nullable\":true,\"metadata\":{}}

In [29]:
import os
import shutil
import zipfile

SOURCE = "/content/trips_delta"
DEST = "/content/drive/MyDrive/BigData/Praktikum4/trips_delta.zip"

# Pastikan folder sumber tersedia
assert os.path.exists(SOURCE), "Folder trips_delta tidak ditemukan!"

# Periksa jumlah log
log_dir = os.path.join(SOURCE, "_delta_log")
logs = [f for f in os.listdir(log_dir) if f.endswith(".json")]

print("Jumlah file JSON:", len(logs))

# Buat ZIP seluruh folder Delta Lake
if os.path.exists(DEST):
    os.remove(DEST)

shutil.make_archive(
    DEST.replace(".zip", ""),
    "zip",
    root_dir="/content",
    base_dir="trips_delta"
)

# Verifikasi isi ZIP
with zipfile.ZipFile(DEST, "r") as z:
    isi_log = [
        f for f in z.namelist()
        if "/_delta_log/" in f and f.endswith(".json")
    ]

print("ZIP berhasil dibuat:", DEST)
print("Jumlah log JSON dalam ZIP:", len(isi_log))
print("Contoh file:", isi_log[:5])

assert len(isi_log) == len(logs), "Ada file log yang tidak ikut terarsip!"
print("VERIFIKASI BERHASIL")

Jumlah file JSON: 25
ZIP berhasil dibuat: /content/drive/MyDrive/BigData/Praktikum4/trips_delta.zip
Jumlah log JSON dalam ZIP: 25
Contoh file: ['trips_delta/_delta_log/00000000000000000015.json', 'trips_delta/_delta_log/00000000000000000020.json', 'trips_delta/_delta_log/00000000000000000008.json', 'trips_delta/_delta_log/00000000000000000006.json', 'trips_delta/_delta_log/00000000000000000019.json']
VERIFIKASI BERHASIL


# S. Pertanyaan Evaluasi

**1. Mengapa NoSQL sering dibaca “Not Only SQL”? Sebutkan dua situasi ketika tabel relasional tetap pilihan yang lebih baik.**

NoSQL disebut “Not Only SQL” karena bukan berarti SQL ditinggalkan sepenuhnya, melainkan terdapat berbagai model penyimpanan lain seperti Key-Value, Document, dan Graph yang dapat digunakan sesuai kebutuhan data.

Tabel relasional tetap lebih baik ketika:
- Data memiliki struktur yang jelas dan membutuhkan hubungan antartabel serta JOIN.
- Transaksi membutuhkan konsistensi tinggi, seperti pembayaran dan pencatatan keuangan, karena SQL mendukung transaksi ACID.

**2. Apa perbedaan BASE dan ACID? Jelaskan teorema CAP dengan satu contoh data pada aplikasi taksi yang sebaiknya memilih consistency, dan satu yang boleh memilih availability.**

ACID menjamin transaksi bersifat atomic, consistent, isolated, dan durable sehingga perubahan data dilakukan secara aman dan konsisten. BASE lebih mengutamakan availability dan fleksibilitas sistem terdistribusi, dengan kemungkinan data menjadi konsisten setelah beberapa waktu (*eventual consistency*).

Teorema CAP menjelaskan bahwa ketika terjadi network partition, sistem terdistribusi harus memilih antara consistency dan availability.

Contohnya:
- **Consistency:** Data pembayaran perjalanan harus konsisten agar tidak terjadi tagihan ganda atau perbedaan nominal pembayaran.
- **Availability:** Lokasi terakhir taksi dapat tetap ditampilkan meskipun sedikit terlambat diperbarui, sehingga aplikasi tetap dapat digunakan.

**3. Mengapa pencarian satu data pada key-value sangat cepat, tetapi menghitung sangat lambat? Apa yang dikorbankan key-value demi kesederhanaannya?**

Key-value sangat cepat karena data dapat diakses langsung menggunakan key tanpa harus memeriksa seluruh data. Sebaliknya, agregasi seperti menghitung rata-rata atau mengelompokkan data membutuhkan pemeriksaan banyak nilai.

Kesederhanaan tersebut mengorbankan fleksibilitas query, kemampuan filtering kompleks, JOIN, dan agregasi langsung. Hal ini terlihat pada K-2, ketika pencarian key-value lebih cepat, tetapi agregasinya lebih lambat dibandingkan SQL.

**4. Jelaskan perbedaan menyatukan data (embedding) dan merujuk (referencing) pada model document, dan satu kondisi yang membuat masing-masing lebih tepat.**

- **Embedding:** Data terkait disimpan langsung di dalam satu dokumen sebagai struktur bersarang. Cocok ketika data sering dibaca bersama dan ukuran dokumennya masih terkendali.
- **Referencing:** Dokumen menyimpan ID atau referensi menuju dokumen lain. Cocok ketika data digunakan bersama oleh banyak dokumen atau sering diperbarui secara terpisah.

Embedding memudahkan pembacaan data terkait, sedangkan referencing mengurangi duplikasi data.

**5. Kapan graph lebih unggul dari SQL untuk pertanyaan keterhubungan? Berikan alasan yang tidak bergantung pada kecepatan.**

Graph lebih unggul ketika pertanyaan berfokus pada hubungan antarentitas dan penelusuran beberapa tingkat keterhubungan, seperti mencari zona yang dapat dicapai dari zona tertentu melalui dua atau tiga perjalanan.

Keunggulannya bukan hanya kecepatan, tetapi karena hubungan direpresentasikan secara langsung sebagai node dan edge. Struktur ini lebih mudah dipahami dan dimodelkan untuk analisis keterhubungan dibandingkan rangkaian JOIN pada SQL.

**6. Apa yang dilakukan indeks spasial sebelum memeriksa lokasi secara tepat? Mengapa klaim “indeks 1.000 kali lebih cepat” bisa menyesatkan?**

Indeks spasial seperti STRtree terlebih dahulu melakukan penyaringan berdasarkan bounding box untuk memperoleh kandidat lokasi yang mungkin berada dalam area pencarian. Setelah itu, pemeriksaan geometri secara tepat dilakukan terhadap kandidat tersebut.

Klaim “1.000 kali lebih cepat” dapat menyesatkan karena hasil pengukuran dipengaruhi ukuran data, jumlah kandidat, ukuran area pencarian, waktu pembangunan indeks, dan jumlah query. Perbandingan juga harus menggunakan jumlah data dan kondisi pengujian yang setara.

**7. Apa isi folder _delta_log/ dan bagaimana ia membuat penulisan bersifat atomik dan time travel mungkin? Mengapa VACUUM dengan retention 0 jam berbahaya?**

Folder `_delta_log/` berisi file JSON yang mencatat riwayat transaksi Delta Lake, seperti informasi commit, metadata, skema, penambahan file (`add`), dan penghapusan file (`remove`).

Setiap transaksi yang berhasil menghasilkan versi log baru. Delta Lake menggunakan commit transaksi untuk memperbarui versi tabel secara atomik, sehingga pembaca melihat kondisi tabel yang konsisten, bukan perubahan setengah jadi. Riwayat versi tersebut memungkinkan Time Travel untuk membaca kondisi tabel pada versi sebelumnya.

VACUUM dengan retention 0 jam berbahaya karena dapat menghapus file Parquet lama yang sudah tidak aktif, padahal file tersebut mungkin masih dibutuhkan untuk membaca versi sebelumnya atau oleh proses yang sedang berjalan. Akibatnya, Time Travel dapat gagal dan data yang masih dibutuhkan bisa hilang.